# Easy Compression with `numcodecs`

In this short course, we use the simple `numcodecs` [^numcodecs] API for compression. In `numcodecs`, compressors are called `Codec`s and only have two important methods: `codec.encode(data)` and `codec.decode(encoded)`. `numcodecs` works great for the simple case of compressing and decompressing in-memory `numpy` arrays in one go, though it can also be used with chunked data.

[^numcodecs]: <https://numcodecs.readthedocs.io/en/stable/>

In [1]:
import numcodecs
import numpy as np

## Configuration

Each codec is a class that inherits from `numcodecs.abc.Codec`. Codecs are configured upon instantiation using keyword arguments. For example, here we create an instance of the lossless `Zlib` compressor:

In [2]:
codec = numcodecs.Zlib(level=6)
codec

Zlib(level=6)

We can obtain the full configuration of the codec in JSON format using `codec.get_config()`:

In [3]:
config = codec.get_config()
config

{'id': 'zlib', 'level': 6}

And then use `numcodecs.get_codec(config)` to recreate the codec from its configuration:

In [4]:
numcodecs.get_codec(config)

Zlib(level=6)

## Compression

We use the `codec.encode(data)` function to encode the data:

In [5]:
data = np.linspace(-10, 10, 41)
data

array([-10. ,  -9.5,  -9. ,  -8.5,  -8. ,  -7.5,  -7. ,  -6.5,  -6. ,
        -5.5,  -5. ,  -4.5,  -4. ,  -3.5,  -3. ,  -2.5,  -2. ,  -1.5,
        -1. ,  -0.5,   0. ,   0.5,   1. ,   1.5,   2. ,   2.5,   3. ,
         3.5,   4. ,   4.5,   5. ,   5.5,   6. ,   6.5,   7. ,   7.5,
         8. ,   8.5,   9. ,   9.5,  10. ])

In [6]:
encoded = codec.encode(data)
encoded

b'x\x9c5\xcc;\x0e@\x00\x14D\xd1W(\x14\n\x11\x11\x11\x11\xbf}\xb0s\x96b\tJ\xa5\xe0\xb8\xcd)&\x99\x88\xa7y\x7f\x89\x89#\x07\xf6\xec\xd8\xb2a\xcd\x8a%\x0b\xe6\xcc\x982a\xf0\xda>O\x1e\xfc;\x16;/\xc6\xea\x8f)3\xe6,X\xb2b\xcd\x86-;\xf6\x1c8r\xe2\xbc\xde>E\x1c\x7f'

## Decompression

We use the `codec.decode(encoded)` function to decode the data:

In [7]:
codec.decode(encoded)

b'\x00\x00\x00\x00\x00\x00$\xc0\x00\x00\x00\x00\x00\x00#\xc0\x00\x00\x00\x00\x00\x00"\xc0\x00\x00\x00\x00\x00\x00!\xc0\x00\x00\x00\x00\x00\x00 \xc0\x00\x00\x00\x00\x00\x00\x1e\xc0\x00\x00\x00\x00\x00\x00\x1c\xc0\x00\x00\x00\x00\x00\x00\x1a\xc0\x00\x00\x00\x00\x00\x00\x18\xc0\x00\x00\x00\x00\x00\x00\x16\xc0\x00\x00\x00\x00\x00\x00\x14\xc0\x00\x00\x00\x00\x00\x00\x12\xc0\x00\x00\x00\x00\x00\x00\x10\xc0\x00\x00\x00\x00\x00\x00\x0c\xc0\x00\x00\x00\x00\x00\x00\x08\xc0\x00\x00\x00\x00\x00\x00\x04\xc0\x00\x00\x00\x00\x00\x00\x00\xc0\x00\x00\x00\x00\x00\x00\xf8\xbf\x00\x00\x00\x00\x00\x00\xf0\xbf\x00\x00\x00\x00\x00\x00\xe0\xbf\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xe0?\x00\x00\x00\x00\x00\x00\xf0?\x00\x00\x00\x00\x00\x00\xf8?\x00\x00\x00\x00\x00\x00\x00@\x00\x00\x00\x00\x00\x00\x04@\x00\x00\x00\x00\x00\x00\x08@\x00\x00\x00\x00\x00\x00\x0c@\x00\x00\x00\x00\x00\x00\x10@\x00\x00\x00\x00\x00\x00\x12@\x00\x00\x00\x00\x00\x00\x14@\x00\x00\x00\x00\x00\x00\x16@\x00\x00\x00\x00\x00\x

Some codecs, like `Zlib`, need to know the type and shape of the original data to correctly decompress it. For these cases, it is also possible to decode directly into an existing array using `codec.decode(encoded, out=decoded)`:

In [8]:
codec.decode(encoded, out=np.empty_like(data))

array([-10. ,  -9.5,  -9. ,  -8.5,  -8. ,  -7.5,  -7. ,  -6.5,  -6. ,
        -5.5,  -5. ,  -4.5,  -4. ,  -3.5,  -3. ,  -2.5,  -2. ,  -1.5,
        -1. ,  -0.5,   0. ,   0.5,   1. ,   1.5,   2. ,   2.5,   3. ,
         3.5,   4. ,   4.5,   5. ,   5.5,   6. ,   6.5,   7. ,   7.5,
         8. ,   8.5,   9. ,   9.5,  10. ])

## Building up more complex compressors

It is often useful to combine several codecs, e.g. to use one to transform the data and one to compress it. Meta-compressors are such flexible combinators, and some are provided in the `numcodecs-combinators` [^numcodecs-combinators] package:

[^numcodecs-combinators]: <https://numcodecs-combinators.readthedocs.io/en/latest/>

In [9]:
from numcodecs_combinators.stack import CodecStack

`CodecStack(a, b, c)` combines codecs `a`, `b`, and `c` into a linear stack. During encoding, the data is first encoded by `a`, then `b`, then `c`. During decoding, the encoded data is first decoded by `c`, then by `b`, then `a`. The `CodecStack` supports an arbitrary number of sub-codecs.

The `CodecStack` also provides two useful helper methods:

- `CodecStack.encode_decode(data)` encodes and then decodes the data array in one go.
- `CodecStack.encode_decode_data_array(da)` encodes and then decodes an `xarray.DataArray` and preserves all metadata. If the data is chunked, the sub-codecs are applied independently to each data chunk.

In [10]:
from numcodecs_combinators.framed import FramedCodecStack

The `FramedCodecStack` is a specialised `CodecStack` that remembers the type and shape of the data during encoding, encodes it alongside the encoded data, and then uses this information during decoding to provide the `out` parameter to each `codec.decode`. The `FramedCodecStack` is thus useful when trying to stack codecs that require this information to decode correctly, or to ensure that any codec produces its encoded data as a byte string.

In [11]:
from numcodecs_combinators.best import PickBestCodec

Finally, the `PickBestCodec(a, b, c)` combinator tries to encode the data with codecs `a`, `b`, or `c`, chooses the one with the greatest reduction in byte size, and outputs its encoded data alongside the index of the codec that was picked. During decoding, the picked sub-codec is used to decode the data. The `PickBestCodec` supports an arbitrary number of sub-codecs.

## Portable scientific compressors with `numcodecs-wasm`

In this short course, we utilize the `numcodecs-wasm` [^numcodecs-wasm] project, which provides several scientific compressors in a `numcodecs`-compatible form. The compressors are compiled to WebAssembly to ensure that compression and decompression can be performed reproducibly across any CPU machine [^climatebenchpress]. Each compressor is published as its own Python package on PyPi, e.g. `numcodecs-wasm-sz3` for the SZ3 compressor.

The complete list of currently supported compressors is:

- [`numcodecs_wasm_asinh`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_asinh): $\text{asinh}(x)$
- [`numcodecs_wasm_bit_round`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_bit_round): bit rounding
- [`numcodecs_wasm_ebcc`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_ebcc): EBCC
- [`numcodecs_wasm_fixed_offset_scale`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_fixed_offset_scale): $\frac{x - o}{s}$
- [`numcodecs_wasm_fourier_network`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_fourier_network): fourier feature neural network
- [`numcodecs_wasm_identity`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_identity): $f(x) = x$
- [`numcodecs_wasm_jpeg2000`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_jpeg2000): JPEG 2000
- [`numcodecs_wasm_lc`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_lc): LC
- [`numcodecs_wasm_linear_quantize`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_linear_quantize): linear quantization
- [`numcodecs_wasm_log`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_log): $\ln(x)$
- [`numcodecs_wasm_pco`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_pco): pcodec
- `numcodecs_wasm_pressio`: Pressio (work-in-progress)
- [`numcodecs_wasm_qpet_sperr`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_qpet_sperr): QPET-SPERR
- [`numcodecs_wasm_random_projection`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_random_projection): random projection
- [`numcodecs_wasm_reinterpret`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_reinterpret): binary reinterpret
- [`numcodecs_wasm_round`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_round): rounding
- [`numcodecs_wasm_sperr`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_sperr): SPERR
- [`numcodecs_wasm_stochastic_rounding`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_stochastic_rounding): stochastic rounding
- [`numcodecs_wasm_swizzle_reshape`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_swizzle_reshape): array axis swizzle and reshape
- [`numcodecs_wasm_sz3`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_sz3): SZ3
- [`numcodecs_wasm_tthresh`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_tthresh): Tthresh
- [`numcodecs_wasm_uniform_noise`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_uniform_noise): add uniform noise
- [`numcodecs_wasm_zfp`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_zfp): ZFP-ROUND
- `numcodecs_wasm_zfp_classic`: ZFP
- [`numcodecs_wasm_zlib`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_zlib): Zlib
- [`numcodecs_wasm_zstd`](https://numcodecs-wasm.readthedocs.io/en/latest/api/numcodecs_wasm_zstd): Zstdandard

While `numcodecs-wasm` provides reproducibility and makes it easy to quickly test out different compressors, it has reduced performance as compressors are run single-threaded in a WebAssembly engine and not as native code. Furthermore, the codecs run inside a sandboxed 32 bit memory space and can typically only compress arrays up to 1-2 GB in size. To compress larger data with `numcodecs-wasm`, please chunk the data using `xarray` and use `CodecStack.encode_decode_data_array(da)`.

[^numcodecs-wasm]: <https://numcodecs-wasm.readthedocs.io/en/latest/>

[^climatebenchpress]: Reichelt, T., Tyree, J., Klöwer, M., Dueben, P., Lawrence, B. N., Baker, A. H., Faghih-Naini, S., Hoefler, T., & Stier, P. (2026). ClimateBenchPress (v1.0): A Benchmark for Lossy Compression of Climate Data. *EGUsphere [Preprint]*. Available from: [doi:10.5194/egusphere-2026-60](https://doi.org/10.5194/egusphere-2026-60).